# 🎙️ Voxtral Mini-4B — Tunisian Arabic ASR Benchmark

**Model:** `mistralai/Voxtral-Mini-4B-Realtime-2602`  
**Config key:** `voxtral`

> Voxtral is a realtime streaming ASR model.  
> We use: `processor(audio_list)` → `model.generate()` → `batch_decode()`  
> Requires `transformers >= 5.2.0`

## 1 · Install & Imports

In [5]:
!pip install -q datasets torchaudio evaluate jiwer pyyaml
!pip install --upgrade transformers
!pip install --upgrade "mistral-common[audio]"



In [6]:
import transformers
transformers.__version__

'5.7.0'

In [1]:
# Local path to benchmarking package
import sys
from pathlib import Path

BENCHMARK_ROOT = Path('/home/ala/dataset')
assert BENCHMARK_ROOT.exists(), f'Missing path: {BENCHMARK_ROOT}'

if str(BENCHMARK_ROOT) not in sys.path:
    sys.path.insert(0, str(BENCHMARK_ROOT))

print('Using benchmark root:', BENCHMARK_ROOT)

Using benchmark root: /home/ala/dataset


In [7]:
from benchmark_utils import (
    load_config, get_device, print_gpu_info, setup_output_dir,
    load_benchmark, split_benchmark,
    compute_metrics, per_sample_wer,
    run_pipeline_inference, build_results_df,
    run_labelled_splits, run_unlabelled_splits,
    display_preview, display_worst, display_bulk_predictions,
    audio_inspector, display_summary, plot_wer_cer,
)
import numpy as np
import torch
import time
from pathlib import Path
from tqdm.auto import tqdm
from datasets import Audio as HFAudio

cfg = load_config(str(BENCHMARK_ROOT / 'config.yaml'))

# Override Colab-only paths for local execution
cfg['paths']['dataset'] = str(BENCHMARK_ROOT)
cfg['paths']['output_root'] = str(Path('/home/ala/TunisianDialogSystem/outputs/asr_benchmark_results'))

TARGET_SR    = cfg['evaluation']['target_sr']
TOP_N_WORST  = cfg['evaluation']['top_n_worst']
PREVIEW_ROWS = cfg['evaluation']['preview_rows']
RESUME = False

print('Dataset path:', cfg['paths']['dataset'])
print('Output root :', cfg['paths']['output_root'])

Dataset path: /home/ala/dataset
Output root : /home/ala/TunisianDialogSystem/outputs/asr_benchmark_results


## 2 · GPU Check

In [5]:
device = get_device()
print_gpu_info()

Device : cuda
GPU    : NVIDIA GB10
VRAM   : 128.5 GB total  |  128.5 GB free


## 3 · Load Model

In [8]:
import torch
from transformers import VoxtralRealtimeForConditionalGeneration, AutoProcessor
from transformers import AutoProcessor

mcfg        = cfg["models"]["voxtral"]
MODEL_ID    = mcfg["model_id"]
BATCH_SIZE  = mcfg["batch_size"]
OUTPUT_DIR  = setup_output_dir(cfg, "voxtral")
FILE_SUFFIX = "voxtral_results"

torch_dtype = torch.float16  # or torch.bfloat16 if your GPU supports it

print(f"Loading {MODEL_ID} ...")

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = VoxtralRealtimeForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto",
)

model.eval()
print(f"✓ {MODEL_ID} loaded")

Loading mistralai/Voxtral-Mini-4B-Realtime-2602 ...


processor_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tekken.json:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.86G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✓ mistralai/Voxtral-Mini-4B-Realtime-2602 loaded


## 4 · Mount Drive & Load Benchmark

In [ ]:
from datasets import DatasetDict, load_from_disk

# Build a DatasetDict from available on-disk split folders only
split_dirs = sorted([p for p in BENCHMARK_ROOT.iterdir() if p.is_dir()])
benchmark = DatasetDict()

for p in split_dirs:
    try:
        benchmark[p.name] = load_from_disk(str(p))
    except Exception:
        pass

print(f"Loaded splits: {list(benchmark.keys())}")
LABELLED_SPLITS, UNLABELLED_SPLITS = split_benchmark(benchmark)

FileNotFoundError: No such files: '/home/ala/dataset/labeled_linagora_cs_keep/dataset_info.json', nor '/home/ala/dataset/labeled_linagora_cs_keep/state.json' found. Expected to load a `Dataset` object but provided path is not a `Dataset`.

## 5 · Inference

In [ ]:
def infer_fn(ds):
    ds = ds.cast_column("audio", HFAudio(sampling_rate=TARGET_SR))

    predictions, latencies = [], []
    start = time.time()

    for i in tqdm(range(0, len(ds), BATCH_SIZE), desc="Inferring", unit="batch"):
        batch = ds.select(range(i, min(i + BATCH_SIZE, len(ds))))

        audio_list = [
            np.array(x["audio"]["array"], dtype=np.float32)
            for x in batch
        ]

        t0 = time.time()

        inputs = processor(
            audio_list,
            sampling_rate=TARGET_SR,
            return_tensors="pt"
        )

        inputs = inputs.to(model.device, dtype=model.dtype)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256)

        texts = processor.batch_decode(outputs, skip_special_tokens=True)

        elapsed = time.time() - t0
        per_s = elapsed / len(audio_list)

        predictions.extend(texts)
        latencies.extend([per_s] * len(texts))

    return predictions, latencies, time.time() - start


all_result_dfs, summary_rows = run_labelled_splits(
    benchmark,
    LABELLED_SPLITS,
    infer_fn,
    OUTPUT_DIR,
    FILE_SUFFIX,
    PREVIEW_ROWS,
    TOP_N_WORST,
)


  Running: labeled_algerian  (400 samples)


Inferring:   0%|          | 0/100 [00:00<?, ?batch/s]

  WER              : 0.8704
  CER              : 0.6440
  RTF              : 0.2274
  Total audio      : 3.175 h
  Inference time   : 2598.9 s
  Mean latency/smp : 6.4570 s
  Median latency   : 6.4041 s
  ✓ CSV saved → /content/drive/My Drive/asr_benchmark_results/voxtral_results/labeled_algerian_voxtral_results.csv

  Running: labeled_linagora_raw  (2380 samples)


Inferring:   0%|          | 0/595 [00:00<?, ?batch/s]

KeyboardInterrupt: 

## 6 · Per-sample preview

In [ ]:
display_preview(all_result_dfs, PREVIEW_ROWS, extra_cols=['detected_language'])

## 7 · Worst predictions

In [ ]:
display_worst(all_result_dfs, TOP_N_WORST, extra_cols=['detected_language'])

## 8 · Unlabelled inspection

In [ ]:
unlabelled_result_dfs = run_unlabelled_splits(
    benchmark,
    UNLABELLED_SPLITS,
    infer_fn,
    OUTPUT_DIR,
    FILE_SUFFIX,
    resume=RESUME,
)

In [ ]:
audio_inspector(
    benchmark,
    unlabelled_result_dfs,
    UNLABELLED_SPLITS,
    target_sr=TARGET_SR,
    extra_df_cols=['detected_language'],
)

In [ ]:
display_bulk_predictions(unlabelled_result_dfs, extra_cols=['detected_language'])

## 9 · Summary

In [ ]:
summary_df = display_summary(
    summary_rows,
    OUTPUT_DIR,
    FILE_SUFFIX,
    "Voxtral-Mini-4B"
)

In [ ]:
if summary_df is not None:
    plot_wer_cer(
        summary_df,
        OUTPUT_DIR,
        FILE_SUFFIX,
        "Voxtral-Mini-4B"
    )